# Shared configuration


In [ ]:
# Set resource limits before importing numerical libraries.
import os
CPU_WORKERS = max(1, int(os.environ.get("CHD_CPU_WORKERS", "4")))
INNER_THREADS = max(1, int(os.environ.get("CHD_INNER_THREADS", "1")))
for _key in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_key, str(INNER_THREADS))

import hashlib, json, platform, re, time, uuid, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceWarning
from lifelines.utils import concordance_index
try:
    from IPython.display import display
except Exception:
    display = print

PROJECT_SCHEMA_VERSION = "2.1.0"
FIELD_SEMANTICS={"4079":"diastolic_blood_pressure","4080":"systolic_blood_pressure"}
PROJECT_NAME = "CHD_PRS_NMR_ServerReady_v2"
RANDOM_STATE = int(os.environ.get("CHD_RANDOM_STATE", "2026"))
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15
REGISTRY_END_DATE_STR = os.environ.get("CHD_REGISTRY_END_DATE", "2021-10-31")
DATA_DIR = os.environ.get("CHD_DATA_DIR", "../data")
file_names = [
    "filtered_ukb_data_251201_1.csv", "filtered_data_251215.csv",
    "filtered_ukb_data_251201_2.csv", "filtered_data_251222.csv",
    "filtered_ukb_data_260110.csv", "ICD9-filtered_ukb_data_260411.csv",
    "filtered_ukb_data_260511.csv",
]
INPUT_FILE_CONTRACT = {
    "required_file_count": 7,
    "ordered_file_names": file_names,
    "missing_file_policy": "fail",
    "duplicate_eid_policy": "fail",
    "duplicate_field_policy": "require equality or fail",
    "merge_key": "Eid",
    "merge_how": "inner",
}
if len(file_names) != 7 or len(set(file_names)) != 7:
    raise ValueError("Input file contract requires seven unique files.")
PRIMARY_COHORT_ID = "prs_raw106_complete"
SENSITIVITY_COHORT_ID = "coverage90"
SENSITIVITY_COHORT_IDS = [SENSITIVITY_COHORT_ID]
CHD_COHORT_ID = os.environ.get("CHD_COHORT_ID", PRIMARY_COHORT_ID).strip()
if CHD_COHORT_ID not in {PRIMARY_COHORT_ID, SENSITIVITY_COHORT_ID}:
    raise ValueError("CHD_COHORT_ID must be prs_raw106_complete or coverage90")
ANALYSIS_ROLE = "primary" if CHD_COHORT_ID == PRIMARY_COHORT_ID else "sensitivity"
PROJECT_OUTPUT_ROOT = Path(os.environ.get(
    "CHD_OUTPUT_ROOT", "../outputs"
))
RUN_LABEL = os.environ.get("CHD_RUN_ID", "final_rerun_2026").strip()
if not RUN_LABEL or not re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_LABEL):
    raise ValueError("Invalid CHD_RUN_ID")
RUN_ID = RUN_LABEL
SENSITIVITY_OUTPUT_DIR = str(PROJECT_OUTPUT_ROOT / "runs" / RUN_ID)
os.environ.setdefault("CHD_SENSITIVITY_OUTPUT_DIR", SENSITIVITY_OUTPUT_DIR)
PIPELINE_ARTIFACT_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "01_cohort")
PREPROCESSING_ARTIFACT_DIR = os.path.join(PIPELINE_ARTIFACT_DIR, "preprocessing")
SCORE_ARTIFACT_DIR = os.path.join(PIPELINE_ARTIFACT_DIR, "score_registry")
COHORT_SENSITIVITY_DIR = os.path.join(PIPELINE_ARTIFACT_DIR, "sensitivity_coverage90")
MODEL_ARTIFACT_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "02_models")
PREDICTION_ARTIFACT_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "03_predictions")
RESULTS_OUTPUT_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "04_results")
TABLE_DIR = os.path.join(RESULTS_OUTPUT_DIR, "tables")
FIGURE_DATA_DIR = os.path.join(RESULTS_OUTPUT_DIR, "figure_source_data")
FIGURE_DIR = os.path.join(RESULTS_OUTPUT_DIR, "figures")
LOG_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "07_logs")
CHECKPOINT_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "08_checkpoints")
AUDIT_DIR = os.path.join(SENSITIVITY_OUTPUT_DIR, "09_audit")
for _directory in [PIPELINE_ARTIFACT_DIR, PREPROCESSING_ARTIFACT_DIR, SCORE_ARTIFACT_DIR,
                   COHORT_SENSITIVITY_DIR, MODEL_ARTIFACT_DIR, PREDICTION_ARTIFACT_DIR,
                   RESULTS_OUTPUT_DIR, TABLE_DIR, FIGURE_DATA_DIR, FIGURE_DIR,
                   LOG_DIR, CHECKPOINT_DIR, AUDIT_DIR]:
    Path(_directory).mkdir(parents=True, exist_ok=True)
MAIN_SCENARIO_NAME = f"incident_chd_{CHD_COHORT_ID}"
EXCLUDE_BASELINE_DIABETES = os.environ.get("CHD_EXCLUDE_BASELINE_DIABETES", "1") == "1"
EXCLUDE_BASELINE_LIPID_LOWERING = os.environ.get("CHD_EXCLUDE_BASELINE_LIPID_LOWERING", "1") == "1"
EXCLUDE_BASELINE_GESTATIONAL_DIABETES_IN_MAIN = False
APPLY_GENETIC_QC = os.environ.get("CHD_APPLY_GENETIC_QC", "1") == "1"
_ALLOWED_22006_RAW = os.environ.get("CHD_ALLOWED_22006_VALUES", "1").strip()
if _ALLOWED_22006_RAW:
    _allowed_22006_tokens=[value.strip() for value in _ALLOWED_22006_RAW.split(",")]
    if any(not value for value in _allowed_22006_tokens):
        raise ValueError("CHD_ALLOWED_22006_VALUES must be comma-separated integers or an empty string")
    try:
        ALLOWED_22006_VALUES=[int(value) for value in _allowed_22006_tokens]
    except ValueError as exc:
        raise ValueError(
            "CHD_ALLOWED_22006_VALUES must be comma-separated integers or an empty string"
        ) from exc
    if len(set(ALLOWED_22006_VALUES))!=len(ALLOWED_22006_VALUES):
        raise ValueError("CHD_ALLOWED_22006_VALUES contains duplicate values")
else:
    # Empty means genetic QC may still run, but Field 22006 ancestry restriction is skipped.
    ALLOWED_22006_VALUES=[]
MAIN_COHORT_CONTRACT_NAME = PRIMARY_COHORT_ID
NMR_COVERAGE90_SENSITIVITY_THRESHOLD = float(os.environ.get("CHD_NMR_COVERAGE90_THRESHOLD", "0.90"))
if not 0.0 < NMR_COVERAGE90_SENSITIVITY_THRESHOLD < 1.0:
    raise ValueError("CHD_NMR_COVERAGE90_THRESHOLD must be between 0 and 1")
SPEARMAN_NMR_THRESHOLD = float(os.environ.get("CHD_SPEARMAN_THRESHOLD", "0.90"))
NMR_ELASTICNET_L1_RATIOS = [float(v) for v in os.environ.get(
    "CHD_NMR_SCORE_L1_RATIOS",
    "1e-6,1e-5,1e-4,1e-3,0.005,0.01,0.025,0.05,0.1,0.25,0.5,0.75,0.9,0.99,1.0"
).split(",")]
if (
    len(NMR_ELASTICNET_L1_RATIOS)<2
    or len(set(NMR_ELASTICNET_L1_RATIOS))!=len(NMR_ELASTICNET_L1_RATIOS)
    or NMR_ELASTICNET_L1_RATIOS!=sorted(NMR_ELASTICNET_L1_RATIOS)
    or any(not 0.0<value<=1.0 for value in NMR_ELASTICNET_L1_RATIOS)
    or not np.isclose(NMR_ELASTICNET_L1_RATIOS[-1],1.0)
):
    raise ValueError(
        "CHD_NMR_SCORE_L1_RATIOS must be unique, increasing, within (0,1], "
        "and end at the legal pure-L1 endpoint 1.0"
    )
NMR_ELASTICNET_ALPHA_GRID = np.logspace(
    float(os.environ.get("CHD_NMR_SCORE_LOG10_ALPHA_MAX", "-1")),
    float(os.environ.get("CHD_NMR_SCORE_LOG10_ALPHA_MIN", "-7")),
    int(os.environ.get("CHD_NMR_SCORE_N_ALPHAS", "80")),
)
NMR_ELASTICNET_MAX_ITER = int(os.environ.get("CHD_NMR_SCORE_MAX_ITER", "100000"))
NMR_ELASTICNET_INNER_CV = int(os.environ.get("CHD_NMR_SCORE_INNER_CV", "5"))
NMR_BOUNDARY_POLICY = os.environ.get("CHD_NMR_SCORE_BOUNDARY_POLICY", "fail").lower()
if NMR_BOUNDARY_POLICY not in {"fail", "warn"}: raise ValueError("NMR boundary policy must be fail or warn")
EVALUATION_HORIZONS = [5.0, 10.0]
BOOTSTRAP_N = int(os.environ.get("CHD_BOOTSTRAP_N", "1000"))
BOOTSTRAP_RANDOM_STATE = int(os.environ.get("CHD_BOOTSTRAP_SEED", str(RANDOM_STATE)))
BOOTSTRAP_PROGRESS_EVERY = 50
COX_PENALIZER = 0.0
DEEPSURV_MODEL_LABEL = "DeepSurv"
CATEGORICAL_FEATURE_COLS = ["31", "smoking_previous", "smoking_current"]
ADD_MISSING_INDICATORS = False
CNS = {"blue":"#3C5488", "orange":"#E64B35", "teal":"#00A087", "purple":"#8491B4",
       "yellow":"#F39B7F", "green":"#4DBBD5", "gray":"#7E7E7E", "light_gray":"#D9D9D9", "black":"#1A1A1A"}
MODEL_COLORS = {"Cox PH":CNS["blue"], "Elastic-net Cox":CNS["teal"],
                "XGBoost survival":CNS["orange"], "DeepSurv":CNS["purple"]}
plt.rcParams.update({"figure.dpi":160, "savefig.dpi":600, "font.family":"DejaVu Sans",
    "font.size":13, "axes.titlesize":15, "axes.labelsize":13.5, "xtick.labelsize":12,
    "ytick.labelsize":12, "legend.fontsize":11, "axes.linewidth":0.9,
    "axes.spines.top":False, "axes.spines.right":False, "lines.linewidth":2.0,
    "pdf.fonttype":42, "ps.fonttype":42, "svg.fonttype":"none"})
print("Project:", PROJECT_NAME, PROJECT_SCHEMA_VERSION)
print("Data:", DATA_DIR)
print("Run output:", SENSITIVITY_OUTPUT_DIR)
print("CPU workers / inner threads:", CPU_WORKERS, INNER_THREADS)

NOTEBOOK_VERSION = "CHD_PRS_NMR_ServerReady_v2"
NMR_COVERAGE90_THRESHOLD=NMR_COVERAGE90_SENSITIVITY_THRESHOLD
SPEARMAN_ABS_THRESHOLD=SPEARMAN_NMR_THRESHOLD
CNS_PALETTE=[CNS["blue"],CNS["orange"],CNS["teal"],CNS["purple"],CNS["green"]]
RESULT1_DIR=os.path.join(RESULTS_OUTPUT_DIR,"result1_cohort")
RESULT1_FIGURE_DIR=os.path.join(RESULT1_DIR,"figures")
RESULT1_SUPPLEMENT_DIR=os.path.join(RESULT1_DIR,"supplementary_figures")
for _directory in [RESULT1_DIR,RESULT1_FIGURE_DIR,RESULT1_SUPPLEMENT_DIR]: Path(_directory).mkdir(parents=True,exist_ok=True)


In [ ]:
# 0.1 Feature definitions

# Raw106 is the strict main-cohort completeness contract.
# Full249 (UKB fields 23400-23648) supplies the score candidate panel.
# Direct Full249 modelling is a separate sensitivity analysis.

NMR_106_FIELD_MAP = {
    "Acetate": 23475, "Acetoacetate": 23476, "Acetone": 23477, "Ala": 23460,
    "Albumin": 23479, "ApoA1": 23440, "ApoB": 23439, "bOHbutyrate": 23474,
    "Cholines": 23436, "Citrate": 23473, "Creatinine": 23478, "DHA": 23450,
    "Gln": 23461, "Glucose": 23470, "Gly": 23462, "GlycA": 23480,
    "HDL_size": 23433, "His": 23463, "Ile": 23465, "LA": 23449,
    "Lactate": 23471, "LDL_size": 23432, "Leu": 23466, "MUFA": 23447,
    "Omega_3": 23444, "Omega_6": 23445, "Phe": 23468,
    "Phosphatidylc": 23437, "Phosphoglyc": 23434, "Pyruvate": 23472,
    "SFA": 23448, "Sphingomyelins": 23438, "Tyr": 23469,
    "Unsaturation": 23443, "Val": 23467, "VLDL_size": 23431,
    "XXL_VLDL_CE": 23485, "XXL_VLDL_FC": 23486, "XXL_VLDL_P": 23481, "XXL_VLDL_PL": 23483, "XXL_VLDL_TG": 23487,
    "XL_VLDL_CE": 23492, "XL_VLDL_FC": 23493, "XL_VLDL_P": 23488, "XL_VLDL_PL": 23490, "XL_VLDL_TG": 23494,
    "L_VLDL_CE": 23499, "L_VLDL_FC": 23500, "L_VLDL_P": 23495, "L_VLDL_PL": 23497, "L_VLDL_TG": 23501,
    "M_VLDL_CE": 23506, "M_VLDL_FC": 23507, "M_VLDL_P": 23502, "M_VLDL_PL": 23504, "M_VLDL_TG": 23508,
    "S_VLDL_CE": 23513, "S_VLDL_FC": 23514, "S_VLDL_P": 23509, "S_VLDL_PL": 23511, "S_VLDL_TG": 23515,
    "XS_VLDL_CE": 23520, "XS_VLDL_FC": 23521, "XS_VLDL_P": 23516, "XS_VLDL_PL": 23518, "XS_VLDL_TG": 23522,
    "IDL_CE": 23527, "IDL_FC": 23528, "IDL_P": 23523, "IDL_PL": 23525, "IDL_TG": 23529,
    "L_LDL_CE": 23534, "L_LDL_FC": 23535, "L_LDL_P": 23530, "L_LDL_PL": 23532, "L_LDL_TG": 23536,
    "M_LDL_CE": 23541, "M_LDL_FC": 23542, "M_LDL_P": 23537, "M_LDL_PL": 23539, "M_LDL_TG": 23543,
    "S_LDL_CE": 23548, "S_LDL_FC": 23549, "S_LDL_P": 23544, "S_LDL_PL": 23546, "S_LDL_TG": 23550,
    "XL_HDL_CE": 23555, "XL_HDL_FC": 23556, "XL_HDL_P": 23551, "XL_HDL_PL": 23553, "XL_HDL_TG": 23557,
    "L_HDL_CE": 23562, "L_HDL_FC": 23563, "L_HDL_P": 23558, "L_HDL_PL": 23560, "L_HDL_TG": 23564,
    "M_HDL_CE": 23569, "M_HDL_FC": 23570, "M_HDL_P": 23565, "M_HDL_PL": 23567, "M_HDL_TG": 23571,
    "S_HDL_CE": 23576, "S_HDL_FC": 23577, "S_HDL_P": 23572, "S_HDL_PL": 23574, "S_HDL_TG": 23578,
}

nmr106_cols = [str(x) for x in NMR_106_FIELD_MAP.values()]
FULL249_FIELD_IDS = list(range(23400, 23649))
full249_cols = [str(value) for value in FULL249_FIELD_IDS]
FULL249_FIELD_CONTRACT = {
    "contract_name": "Full249",
    "field_id_start_inclusive": 23400,
    "field_id_end_inclusive": 23648,
    "field_count": 249,
    "naming_rule": "Use Full249; this is a contiguous UKB field contract, not a raw representation.",
    "source_definition": "UKB standard NMR 249-panel contract: 168 absolute measures plus 81 ratios",
    "source_definition_version": "Full249-v1; Category 220/phase3 companion contract verified 2026-08-24",
    "explicitly_excluded_extension_fields": [20280, 20281],
    "explicitly_excluded_extension_meanings": ["glucose-lactate", "spectrometer-corrected alanine"],
    "ordered_field_ids": FULL249_FIELD_IDS,
}
PRS_COL = "26227"

if len(NMR_106_FIELD_MAP) != 106 or len(set(NMR_106_FIELD_MAP.values())) != 106:
    raise ValueError("NMR_106_FIELD_MAP must contain 106 unique fields.")
if len(FULL249_FIELD_IDS) != 249 or len(set(FULL249_FIELD_IDS)) != 249:
    raise ValueError("Full249 field contract must contain 249 unique fields.")
if not set(NMR_106_FIELD_MAP.values()).issubset(FULL249_FIELD_IDS):
    raise ValueError("Raw106 must be a subset of the Full249 field contract.")

basic_ids = [31, 21003, 20116, 21001]
bp_ids = [4079, 4080]
blood_ids = [30690, 30760, 30700, 30720]
metabo_ids = list(FULL249_FIELD_IDS)
prs_ids = [26227]

feature_groups = {
    "basic": basic_ids,
    "bp": bp_ids,
    "blood": blood_ids,
    "metabo_full249_contract": metabo_ids,
    "prs": prs_ids,
}

genetic_pc_id = 22009
pc_numbers = list(range(1, 11))
use_genetic_pcs = True


In [ ]:
# 0.2 Exclusion / QC definitions

#

target_icd10_prefixes = ["I20", "I21", "I22", "I23", "I24", "I25"]
target_icd9_prefixes = ["410", "411", "412", "413", "414"]

baseline_chd_self_report_codes = [1074, 1075]

# Field 2443: doctor diagnosed diabetes.
diabetes_yes_values_2443 = [1]

# UKB Data-Coding 6:
# 1220 diabetes, 1222 type 1 diabetes, 1223 type 2 diabetes.
# 1221 gestational diabetes is kept for conservative sensitivity, not main analysis.
baseline_diabetes_self_report_codes = [1220, 1222, 1223]
baseline_diabetes_self_report_codes_conservative = [1220, 1221, 1222, 1223]

# Fields 6153/6177: medication for cholesterol, blood pressure, diabetes, etc.
# Code 1 = cholesterol lowering medication.
lipid_lowering_values_6153_6177 = [1]

lipid_lowering_codes_20003_main = [
    # general
    1140861922,  # lipid lowering drug

    # statins
    1140861958,  # simvastatin
    1140888594,  # fluvastatin
    1140888648,  # pravastatin
    1141146234,  # atorvastatin
    1141146138,  # lipitor 10mg tablet
    1141192410,  # rosuvastatin
    1141192414,  # crestor 10mg tablet

    # ezetimibe
    1141192736,  # ezetimibe
    1141192740,  # ezetrol 10mg tablet

    # fibrates
    1140861856,  # gemfibrozil
    1140861858,  # lopid 300 capsule
    1140861924,  # bezafibrate
    1140861926,  # bezalip 200mg tablet
    1140861928,  # bezalip-mono 400mg m/r tablet
    1140861954,  # fenofibrate
    1140862026,  # ciprofibrate
    1140862028,  # modalim 100mg tablet

    # bile acid sequestrants
    1140861936,  # questran 4g/sachet powder
    1140861942,  # cholestyramine+aspartame
    1140865576,  # cholestyramine
    1140888590,  # colestipol
]
lipid_lowering_codes_20003 = lipid_lowering_codes_20003_main

# Genetic-QC switches are single-sourced from the environment-aware configuration cell.

qc_indicator_ids = [22019, 22027]
reported_sex_ids = [31]

# Main-analysis exclusion switches are single-sourced in the environment-aware configuration cell.
# If EXCLUDE_BASELINE_GESTATIONAL_DIABETES_IN_MAIN is later set True,
# Field 20002 code 1221 can be included as a stricter sensitivity analysis.


In [ ]:
def find_eid_col(headers):
    for value in headers:
        if str(value).strip().replace('"','') in {"Eid","eid","EID","f.eid"}: return value
    return None

def parse_ukb_col(column):
    match = re.match(r"^(\d+)-(\d+)\.(\d+)$", str(column).strip().replace('"',''))
    return tuple(map(int, match.groups())) if match else None

def get_cols_by_fid(frame, fid, instance=0):
    cols = [c for c in frame.columns if parse_ukb_col(c) and parse_ukb_col(c)[:2] == (int(fid),int(instance))]
    return sorted(cols, key=lambda c: parse_ukb_col(c)[2])

def get_cols_by_fid_any_instance(frame, fid):
    cols = [c for c in frame.columns if parse_ukb_col(c) and parse_ukb_col(c)[0] == int(fid)]
    return sorted(cols, key=lambda c: parse_ukb_col(c)[1:])

def any_value_in_cols(frame, columns, target_values):
    result = pd.Series(False, index=frame.index); targets = {str(v) for v in target_values}
    for column in columns:
        result |= frame[column].astype(str).str.replace(r"\.0$","",regex=True).isin(targets)
    return result

def indicator_abnormal_mask(frame, fid):
    result = pd.Series(False, index=frame.index)
    for column in get_cols_by_fid(frame, fid, 0):
        numeric = pd.to_numeric(frame[column], errors="coerce")
        raw = frame[column].astype(str).str.strip().str.lower()
        result |= (numeric.notna() & (numeric != 0)) | (numeric.isna() & frame[column].notna() & ~raw.isin(["","nan","none","0","0.0"]))
    return result

def rowwise_min_datetime(series_list, index):
    values = [pd.to_datetime(series, errors="coerce") for series in series_list if series is not None]
    return pd.concat(values, axis=1).min(axis=1) if values else pd.Series(pd.NaT,index=index,dtype="datetime64[ns]")

def get_first_event_date_from_icd(frame, code_fid, date_fid, target_prefixes):
    has_event = pd.Series(False,index=frame.index)
    first_date = pd.Series(pd.NaT,index=frame.index,dtype="datetime64[ns]")
    for code_column in get_cols_by_fid(frame, code_fid, 0):
        date_column = code_column.replace(str(code_fid),str(date_fid),1)
        if date_column not in frame.columns: continue
        value = frame[code_column].astype("string").str.upper().str.strip()
        if int(code_fid)==41271: value=value.str.replace(".0","",regex=False).str.replace(".","",regex=False)
        matched=value.str.startswith(tuple(target_prefixes),na=False)
        current=pd.to_datetime(frame[date_column],errors="coerce")
        update=matched & current.notna() & (first_date.isna() | (current<first_date))
        first_date.loc[update]=current.loc[update]; has_event |= matched
    return has_event, first_date

def sha256_file(path, chunk_size=1024*1024):
    digest=hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda:handle.read(chunk_size),b""): digest.update(chunk)
    return digest.hexdigest()

def canonical_json_digest(payload):
    return hashlib.sha256(json.dumps(payload,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()).hexdigest()

def vector_digest(values):
    array=np.ascontiguousarray(np.asarray(values)); return hashlib.sha256(array.view(np.uint8)).hexdigest()

def eid_digest(frame):
    return hashlib.sha256("\n".join(frame["Eid"].astype(str)).encode()).hexdigest()

def dataframe_digest(frame, columns=None):
    columns=list(frame.columns if columns is None else columns)
    hashed=pd.util.hash_pandas_object(frame[columns],index=False,categorize=True).to_numpy(np.uint64)
    digest=hashlib.sha256(); digest.update(json.dumps(columns,ensure_ascii=False).encode()); digest.update(np.ascontiguousarray(hashed).view(np.uint8))
    return digest.hexdigest()

def atomic_write_json(payload,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); temporary=path.parent/f".{path.name}.{uuid.uuid4().hex}.tmp"
    try: temporary.write_text(json.dumps(payload,ensure_ascii=False,indent=2),encoding="utf-8"); os.replace(temporary,path)
    finally:
        if temporary.exists(): temporary.unlink()

def atomic_write_csv(frame,path,index=False):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); temporary=path.parent/f".{path.name}.{uuid.uuid4().hex}.tmp"
    try: frame.to_csv(temporary,index=index); os.replace(temporary,path)
    finally:
        if temporary.exists(): temporary.unlink()

def atomic_save_npz(path,**arrays):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True); temporary=path.parent/f".{path.stem}.{uuid.uuid4().hex}.npz"
    try: np.savez_compressed(temporary,**arrays); os.replace(temporary,path)
    finally:
        if temporary.exists(): temporary.unlink()

def safe_filename(value): return re.sub(r"[^A-Za-z0-9_.-]+","_",str(value)).strip("_")
PROGRESS_LOG=os.path.join(LOG_DIR,"running_progress_log.txt")
def log_progress(message):
    line=f"{pd.Timestamp.now()} | {message}"; print(line,flush=True)
    with open(PROGRESS_LOG,"a",encoding="utf-8") as handle: handle.write(line+"\n")
def save_figure(fig,directory,stem):
    paths=[]; Path(directory).mkdir(parents=True,exist_ok=True)
    for extension in ["png","pdf","svg"]:
        path=Path(directory)/f"{stem}.{extension}"; kwargs={"bbox_inches":"tight","facecolor":"white"}
        if extension=="png": kwargs["dpi"]=600
        fig.savefig(path,**kwargs); paths.append(str(path))
    return paths


In [ ]:
def split_train_val_test(frame, scenario_name=MAIN_SCENARIO_NAME):
    train, temporary = train_test_split(frame,train_size=TRAIN_SIZE,random_state=RANDOM_STATE,stratify=frame["Event"])
    validation, test = train_test_split(temporary,test_size=TEST_SIZE/(VAL_SIZE+TEST_SIZE),random_state=RANDOM_STATE,stratify=temporary["Event"])
    outputs=[train.reset_index(drop=True),validation.reset_index(drop=True),test.reset_index(drop=True)]
    for name,part in zip(["train","val","test"],outputs):
        print(scenario_name,name,"N=",len(part),"events=",int(part["Event"].sum()),flush=True)
    return tuple(outputs)

def prepare_X_train_val_test(train_frame,val_frame,test_frame,selected_cols,scale=True):
    """Train-fitted preprocessing without iterative DataFrame insertion.

    Imputation and scaling are unchanged statistically.  Each transformed block
    is materialized once and blocks are concatenated once, which avoids pandas
    fragmentation on Raw106/Full249 feature sets.
    """
    selected_cols=list(selected_cols)
    missing=[column for column in selected_cols if column not in train_frame.columns]
    if missing: raise KeyError(f"Prespecified columns missing: {missing}")
    raws=[part.loc[:,selected_cols].replace([np.inf,-np.inf],np.nan).astype(float) for part in [train_frame,val_frame,test_frame]]
    all_missing=raws[0].columns[raws[0].notna().sum(axis=0)==0].tolist()
    if all_missing: raise ValueError(f"Training columns entirely missing: {all_missing}")
    categorical=[c for c in selected_cols if c in CATEGORICAL_FEATURE_COLS]
    continuous=[c for c in selected_cols if c not in categorical]
    blocks=[[] for _ in raws]
    for columns,strategy in [(continuous,"median"),(categorical,"most_frequent")]:
        if not columns: continue
        imputer=SimpleImputer(strategy=strategy)
        transformed=[imputer.fit_transform(raws[0][columns]),imputer.transform(raws[1][columns]),imputer.transform(raws[2][columns])]
        for split_blocks,value,index in zip(blocks,transformed,[part.index for part in raws]):
            split_blocks.append(pd.DataFrame(value,columns=columns,index=index))
    outputs=[pd.concat(split_blocks,axis=1).loc[:,selected_cols].copy() for split_blocks in blocks]
    constant=outputs[0].columns[outputs[0].nunique(dropna=False)<=1].tolist()
    if constant: raise ValueError(f"Prespecified columns constant in train: {constant}")
    if scale:
        scaler=StandardScaler()
        transformed=[scaler.fit_transform(outputs[0]),scaler.transform(outputs[1]),scaler.transform(outputs[2])]
        outputs=[pd.DataFrame(np.ascontiguousarray(value,dtype=np.float64),columns=selected_cols,index=part.index) for value,part in zip(transformed,outputs)]
    return outputs[0],outputs[1],outputs[2],selected_cols

def calc_cindex_from_risk(frame,risk):
    return float(concordance_index(frame["Time"].astype(float),-np.asarray(risk).reshape(-1),frame["Event"].astype(int)))

def make_result_row(scenario_name,model_name,feature_set_name,selected_cols_actual,train_risk,val_risk,test_risk):
    return {"Scenario":scenario_name,"Feature_Set":feature_set_name,"Model":model_name,
        "Feature_N":int(len(selected_cols_actual)),"NMR_N":int(sum(c in set(nmr106_cols) for c in selected_cols_actual)),
        "Train_N":int(len(df_train)),"Train_Case":int(df_train["Event"].sum()),
        "Val_N":int(len(df_val)),"Val_Case":int(df_val["Event"].sum()),
        "Test_N":int(len(df_test)),"Test_Case":int(df_test["Event"].sum()),
        "Train_C_index":calc_cindex_from_risk(df_train,train_risk),
        "Val_C_index":calc_cindex_from_risk(df_val,val_risk),
        "Test_C_index":calc_cindex_from_risk(df_test,test_risk),
        "Risk_Scale":"log_relative_hazard; higher means greater risk"}


In [ ]:
PIPELINE_MANIFEST_PATH=os.path.join(PIPELINE_ARTIFACT_DIR,"pipeline_manifest.json")
FEATURE_SETS_PATH=os.path.join(PIPELINE_ARTIFACT_DIR,"feature_sets.json")
def split_artifact_path(split_name): return os.path.join(PIPELINE_ARTIFACT_DIR,f"df_{split_name}_enriched.csv")
def model_result_dir(model_name):
    path=os.path.join(MODEL_ARTIFACT_DIR,safe_filename(model_name)); Path(path).mkdir(parents=True,exist_ok=True); return path
def model_result_path(model_name,feature_set_name): return os.path.join(model_result_dir(model_name),f"{safe_filename(feature_set_name)}_result.csv")
def risk_record_path(model_name,feature_set_name): return os.path.join(PREDICTION_ARTIFACT_DIR,safe_filename(model_name),f"{safe_filename(feature_set_name)}_risks.npz")
def risk_metadata_path(model_name,feature_set_name): return os.path.join(PREDICTION_ARTIFACT_DIR,safe_filename(model_name),f"{safe_filename(feature_set_name)}_metadata.json")
def current_pipeline_sha256(): return sha256_file(PIPELINE_MANIFEST_PATH)

def eid_storage_array(frame):
    """Return a pickle-free, fixed-width Unicode EID array for NPZ storage."""
    return np.asarray(frame["Eid"].astype(str).tolist(),dtype=np.str_)

def npz_eid_matches_without_pickle(payload,key,frame,signature_payload,split_name):
    """Validate new Unicode EIDs or a legacy object-EID cache without unpickling.

    Legacy object arrays cannot be opened with allow_pickle=False. Their EID
    order is instead checked against the signed EID digest already stored in
    the cache metadata; callers then rewrite the NPZ using Unicode EIDs.
    """
    try:
        saved=np.asarray(payload[key]).astype(str)
    except ValueError as error:
        if "Object arrays cannot be loaded when allow_pickle=False" not in str(error):
            raise
        return signature_payload.get(f"{split_name}_eid_sha256")==eid_digest(frame),True
    expected=frame["Eid"].astype(str).to_numpy()
    return np.array_equal(saved,expected),False

def model_signature_payload(model_name,feature_set_name,actual_columns,hyperparameters,code_tag):
    return {"schema_version":PROJECT_SCHEMA_VERSION,"run_id":RUN_ID,"scenario":MAIN_SCENARIO_NAME,
        "pipeline_manifest_sha256":current_pipeline_sha256(),"model_name":model_name,"feature_set":feature_set_name,
        "ordered_feature_columns":list(actual_columns),"feature_columns_sha256":canonical_json_digest(list(actual_columns)),
        "hyperparameters":hyperparameters,"code_tag":code_tag,"seed":RANDOM_STATE,
        "train_eid_sha256":eid_digest(df_train),"val_eid_sha256":eid_digest(df_val),"test_eid_sha256":eid_digest(df_test),
        "train_outcome_sha256":dataframe_digest(df_train,["Eid","Time","Event"]),
        "val_outcome_sha256":dataframe_digest(df_val,["Eid","Time","Event"]),
        "test_outcome_sha256":dataframe_digest(df_test,["Eid","Time","Event"]),
        "python_version":platform.python_version(),"risk_direction":"higher_is_higher_hazard"}

def risk_cache_matches(model_name,feature_set_name,actual_columns,hyperparameters,code_tag):
    risk_path=Path(risk_record_path(model_name,feature_set_name)); meta_path=Path(risk_metadata_path(model_name,feature_set_name)); result_path=Path(model_result_path(model_name,feature_set_name))
    if not risk_path.exists() or not meta_path.exists() or not result_path.exists(): return False
    expected=model_signature_payload(model_name,feature_set_name,actual_columns,hyperparameters,code_tag)
    saved=json.loads(meta_path.read_text(encoding="utf-8")); saved_payload=saved.get("signature_payload",{})
    if saved.get("signature")!=canonical_json_digest(saved_payload): return False
    if any(saved_payload.get(key)!=value for key,value in expected.items()): return False
    risk_arrays={}; legacy_object_eid=False
    try:
        with np.load(risk_path,allow_pickle=False) as payload:
            for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
                eid_ok,legacy=npz_eid_matches_without_pickle(payload,f"{split_name}_eid",frame,saved_payload,split_name)
                if not eid_ok: return False
                legacy_object_eid |= legacy
                risk=np.asarray(payload[f"{split_name}_risk"],dtype=np.float64)
                if risk.ndim!=1 or len(risk)!=len(frame) or not np.isfinite(risk).all() or np.std(risk)<=0: return False
                if vector_digest(risk)!=saved_payload.get(f"{split_name}_risk_sha256"): return False
                risk_arrays[split_name]=risk.copy()
    except Exception: return False
    if legacy_object_eid:
        atomic_save_npz(risk_path,
            train_eid=eid_storage_array(df_train),val_eid=eid_storage_array(df_val),test_eid=eid_storage_array(df_test),
            train_risk=risk_arrays["train"],val_risk=risk_arrays["val"],test_risk=risk_arrays["test"])
        log_progress(f"MIGRATED legacy object-EID NPZ | {model_name} | {feature_set_name}")
    return True

def save_risk_record(model_name,feature_set_name,risks,actual_columns=None,hyperparameters=None,code_tag="unspecified"):
    if actual_columns is None: actual_columns=list(final_feature_sets[feature_set_name])
    if hyperparameters is None: hyperparameters={}
    risk_arrays={split_name:np.asarray(risks[f"{split_name}_risk"],dtype=np.float64).reshape(-1) for split_name in ["train","val","test"]}
    for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
        risk=risk_arrays[split_name]
        if len(risk)!=len(frame) or not np.isfinite(risk).all() or np.std(risk)<=0: raise ValueError(f"Invalid {split_name} risk vector for {model_name}|{feature_set_name}")
    payload=model_signature_payload(model_name,feature_set_name,actual_columns,hyperparameters,code_tag)
    payload.update({f"{split_name}_risk_sha256":vector_digest(risk_arrays[split_name]) for split_name in ["train","val","test"]})
    signature=canonical_json_digest(payload)
    atomic_save_npz(risk_record_path(model_name,feature_set_name),
        train_eid=eid_storage_array(df_train),val_eid=eid_storage_array(df_val),test_eid=eid_storage_array(df_test),
        train_risk=risk_arrays["train"],val_risk=risk_arrays["val"],test_risk=risk_arrays["test"])
    atomic_write_json({"signature":signature,"signature_payload":payload},risk_metadata_path(model_name,feature_set_name))
    return signature
def validate_pipeline_artifacts(load_frames=True):
    if not Path(PIPELINE_MANIFEST_PATH).exists() or not Path(FEATURE_SETS_PATH).exists(): raise FileNotFoundError("Run Result 1 first")
    manifest=json.loads(Path(PIPELINE_MANIFEST_PATH).read_text(encoding="utf-8")); feature_sets=json.loads(Path(FEATURE_SETS_PATH).read_text(encoding="utf-8"))
    required={"Clinical_only","Clinical_plus_PRS","Clinical_plus_NMR_score","Clinical_plus_PRS_plus_NMR_score",
              "Clinical_plus_PRS_plus_SpearmanNMR","Clinical_plus_PRS_plus_Raw106NMR"}
    if required.difference(feature_sets): raise RuntimeError(f"Missing feature sets: {sorted(required.difference(feature_sets))}")
    if manifest.get("schema_version")!=PROJECT_SCHEMA_VERSION or manifest.get("scenario")!=MAIN_SCENARIO_NAME: raise RuntimeError("Pipeline schema/scenario mismatch")
    if not load_frames: return None,None,None,feature_sets,manifest
    frames={}
    for split_name in ["train","val","test"]:
        path=split_artifact_path(split_name)
        if sha256_file(path)!=manifest["splits"][split_name]["file_sha256"]: raise RuntimeError(f"{split_name} file hash mismatch")
        frame=pd.read_csv(path,low_memory=False); expected=manifest["splits"][split_name]
        if len(frame)!=expected["n"] or eid_digest(frame)!=expected["eid_sha256"]: raise RuntimeError(f"{split_name} population mismatch")
        if dataframe_digest(frame,["Eid","Time","Event"])!=expected["outcome_sha256"]: raise RuntimeError(f"{split_name} outcome mismatch")
        frames[split_name]=frame
    id_sets=[set(frames[name]["Eid"].astype(str)) for name in ["train","val","test"]]
    if id_sets[0]&id_sets[1] or id_sets[0]&id_sets[2] or id_sets[1]&id_sets[2]: raise RuntimeError("Split EID overlap")
    return frames["train"],frames["val"],frames["test"],feature_sets,manifest

def load_all_risk_records(feature_sets,model_names):
    store={}
    for feature_set_name in feature_sets:
        for model_name in model_names:
            path=Path(risk_record_path(model_name,feature_set_name)); meta=Path(risk_metadata_path(model_name,feature_set_name))
            if not path.exists() or not meta.exists(): raise FileNotFoundError(f"Missing signed risk artifact: {path}")
            metadata=json.loads(meta.read_text(encoding="utf-8")); signature_payload=metadata.get("signature_payload",{})
            if metadata.get("signature")!=canonical_json_digest(signature_payload): raise RuntimeError(f"Invalid metadata signature: {meta}")
            if signature_payload.get("pipeline_manifest_sha256")!=current_pipeline_sha256(): raise RuntimeError(f"Stale risk artifact: {path}")
            record={}; legacy_object_eid=False
            with np.load(path,allow_pickle=False) as payload:
                for split_name,frame in [("train",df_train),("val",df_val),("test",df_test)]:
                    if signature_payload.get(f"{split_name}_eid_sha256")!=eid_digest(frame): raise RuntimeError(f"Signed EID digest mismatch: {model_name}|{feature_set_name}|{split_name}")
                    if signature_payload.get(f"{split_name}_outcome_sha256")!=dataframe_digest(frame,["Eid","Time","Event"]): raise RuntimeError(f"Signed outcome mismatch: {model_name}|{feature_set_name}|{split_name}")
                    eid_ok,legacy=npz_eid_matches_without_pickle(payload,f"{split_name}_eid",frame,signature_payload,split_name)
                    if not eid_ok: raise RuntimeError(f"EID order mismatch: {model_name}|{feature_set_name}|{split_name}")
                    legacy_object_eid |= legacy
                    risk=np.asarray(payload[f"{split_name}_risk"],dtype=float)
                    if risk.ndim!=1 or len(risk)!=len(frame) or not np.isfinite(risk).all(): raise RuntimeError("Invalid risk vector")
                    if vector_digest(np.asarray(risk,dtype=np.float64))!=signature_payload.get(f"{split_name}_risk_sha256"): raise RuntimeError(f"Risk SHA mismatch: {model_name}|{feature_set_name}|{split_name}")
                    record[f"{split_name}_risk"]=risk.copy()
            if legacy_object_eid:
                atomic_save_npz(path,
                    train_eid=eid_storage_array(df_train),val_eid=eid_storage_array(df_val),test_eid=eid_storage_array(df_test),
                    train_risk=record["train_risk"],val_risk=record["val_risk"],test_risk=record["test_risk"])
                log_progress(f"MIGRATED legacy object-EID NPZ | {model_name} | {feature_set_name}")
            record["metadata"]=metadata; store[(MAIN_SCENARIO_NAME,feature_set_name,model_name)]=record
    return store

def fit_unpenalized_cox_feature_sets(train_frame,val_frame,test_frame,feature_sets,feature_names):
    global df_train,df_val,df_test,final_feature_sets
    df_train,df_val,df_test=train_frame,val_frame,test_frame; final_feature_sets=feature_sets; rows=[]
    for feature_set_name in feature_names:
        X_train,X_val,X_test,actual_cols=prepare_X_train_val_test(train_frame,val_frame,test_frame,feature_sets[feature_set_name],scale=True)
        hyperparameters={"penalizer":0.0,"l1_ratio":0.0,"robust_se":True,"ties":"efron"}; code_tag="unpenalized_cox_v2"
        if risk_cache_matches("Cox PH",feature_set_name,actual_cols,hyperparameters,code_tag):
            log_progress(f"RESUME Cox PH | {feature_set_name} | signed cache valid")
            rows.append(pd.read_csv(model_result_path("Cox PH",feature_set_name)).iloc[0].to_dict()); continue
        fit_frame=pd.concat([train_frame[["Time","Event"]].reset_index(drop=True),X_train.reset_index(drop=True)],axis=1)
        log_progress(f"START Cox PH | {feature_set_name} | features={len(actual_cols)}")
        model=CoxPHFitter(penalizer=0.0,l1_ratio=0.0)
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            model.fit(fit_frame,duration_col="Time",event_col="Event",robust=True,show_progress=False)
        risks={"train_risk":model.predict_log_partial_hazard(X_train).to_numpy().reshape(-1),
               "val_risk":model.predict_log_partial_hazard(X_val).to_numpy().reshape(-1),
               "test_risk":model.predict_log_partial_hazard(X_test).to_numpy().reshape(-1)}
        signature=save_risk_record("Cox PH",feature_set_name,risks,actual_cols,hyperparameters,code_tag)
        row=make_result_row(MAIN_SCENARIO_NAME,"Cox PH",feature_set_name,actual_cols,risks["train_risk"],risks["val_risk"],risks["test_risk"])
        row.update({"COX_PENALIZER":0.0,"Robust_SE":True,"Model_Signature":signature})
        atomic_write_csv(pd.DataFrame([row]),model_result_path("Cox PH",feature_set_name),index=False)
        summary=model.summary.reset_index(); summary=summary.rename(columns={summary.columns[0]:"Variable"}); summary.insert(0,"Feature_Set",feature_set_name)
        atomic_write_csv(summary,os.path.join(model_result_dir("Cox PH"),f"{safe_filename(feature_set_name)}_coefficients.csv"),index=False)
        atomic_write_csv(model.baseline_cumulative_hazard_.reset_index(),os.path.join(model_result_dir("Cox PH"),f"{safe_filename(feature_set_name)}_baseline_cumulative_hazard.csv"),index=False)
        rows.append(row); log_progress(f"DONE Cox PH | {feature_set_name} | test Harrell C={row['Test_C_index']:.6f}")
    return pd.DataFrame(rows)
risk_store={}
